In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

# 2. Load and preprocess data
data = pd.read_csv("energy consumption.csv")

data["Timestamp"] = pd.to_datetime(data["Timestamp"], dayfirst=True)
data = data.sort_values("Timestamp").reset_index(drop=True)

features = ["Electricity_Consumed", "Temperature", "Humidity",
            "Wind_Speed", "Avg_Past_Consumption"]

data[features] = data[features].fillna(data[features].median())
data["Anomaly_Label"] = data["Anomaly_Label"].fillna(0)

# Split chronologically: first 80% for training, last 20% for testing
split = int(len(data) * 0.8)
train_data = data.iloc[:split]
test_data = data.iloc[split:]

# Train scaler only on normal training records
scaler = MinMaxScaler()
scaler.fit(train_data.loc[train_data["Anomaly_Label"] == 0, features])

X_train = scaler.transform(train_data[features])
X_test = scaler.transform(test_data[features])

# 3. Create time windows
window_size = 10

def create_windows(X, labels=None, normal_only=False):
    windows, targets = [], []

    for i in range(len(X) - window_size + 1):
        window = X[i:i + window_size]

        if labels is not None:
            label = labels[i:i + window_size]

            if normal_only and np.any(label == 1):
                continue

            targets.append(int(np.any(label == 1)))

        windows.append(window)

    if labels is None:
        return np.array(windows)

    return np.array(windows), np.array(targets)

X_train_seq, _ = create_windows(
    X_train, train_data["Anomaly_Label"].values, normal_only=True
)

X_test_seq, y_test = create_windows(
    X_test, test_data["Anomaly_Label"].values
)

# 4. Build LSTM Autoencoder
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(window_size, len(features))),
    tf.keras.layers.LSTM(32, activation="relu"),
    tf.keras.layers.RepeatVector(window_size),
    tf.keras.layers.LSTM(32, activation="relu", return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(features)))
])

# 5. Compile and train model
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(
    X_train_seq, X_train_seq,
    epochs=20, batch_size=32,
    validation_split=0.1
)

# 6. Calculate reconstruction error and threshold
train_pred = model.predict(X_train_seq)
train_error = np.mean(np.square(X_train_seq - train_pred), axis=(1,2))

threshold = np.percentile(train_error, 95)
print("Threshold:", threshold)

# 7. Detect anomalies
test_pred = model.predict(X_test_seq)
test_error = np.mean(np.square(X_test_seq - test_pred), axis=(1,2))

y_pred = (test_error > threshold).astype(int)

# 8. Evaluate model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# 9. Visualize training loss
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# 10. Visualize reconstruction errors
plt.plot(test_error, label="Reconstruction Error")
plt.axhline(threshold, color="red", linestyle="--", label="Threshold")
plt.xlabel("Time Window")
plt.ylabel("Reconstruction Error")
plt.legend()
plt.show()